# Completez por tramos y recurrencia Gutenberg--Richter

Este notebook ejecuta primero la completez y recurrencia Gutenberg--Richter local (50 km) y regional (260 km), y después la atenuación. Para umbrales Mw < 4.5 estima la ventana reciente con ajuste segmentado; desde Mw 4.5 asigna la duración total del catálogo. Conserva todos los umbrales y marca como referencia cruda los umbrales bajos sin completez estimable. El algoritmo está en `analisis_recurrencia.py`.

Intervalos usados: [3, 3.5), [3.5, 4), [4, 4.5), [4.5, 5), [5, 5.5), [5.5, 6), [6, 6.5), [6.5, 7) y [7, 7.5). Los límites son cerrados por la izquierda y abiertos por la derecha. Se omite el año parcial 2026. La escala local se obtiene del catálogo clasificado limitando la distancia epicentral a 50 km; la regional usa el radio de 260 km.

## Flujo y columnas calculadas

1. Se valida Mw, fecha y sismofuente, y se retienen años calendario completos.
2. Para umbrales Mw < 4.5 se ajustan dos rectas a los acumulados; el quiebre minimiza el error cuadrático y requiere al menos 10 años y 3 eventos por segmento. Se acepta la ventana reciente si R² posterior ≥ 0.90; con N<10 se identifica como aproximada. Desde Mw 4.5 se usa explícitamente todo el catálogo (1917–2025), sin truncar esos puntos. Si no se estima una ventana reciente para Mw < 4.5, se conserva Nacum/T como referencia cruda, sin presentarla como corregida.
3. Por umbral se calculan N incremental, N acumulado, ventana t, tasa sin corrección λcruda=Nacum/T y tasa corregida λ=Nacum/Δtcompleto. Se conserva cada categoría; log10(0) queda indefinido y se identifica aparte en la figura. Una única regresión log10(λ)=a−bM se ajusta con todas las tasas corregidas positivas disponibles, incluidas las de Mw ≥ 4.5; se requieren tres puntos.
4. Se generan series acumuladas, tablas, comparaciones por fuente y escala, y curvas continuas. Los círculos representan tasas sin corrección, los cuadrados tasas corregidas; ecuación y R² identifican cada ajuste.

Los archivos de salida conservan los conteos y métricas que sustentan las figuras y los ajustes.

In [ ]:
from pathlib import Path
import importlib.util
import sys

cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
script_path = next(
    (
        candidate
        for root in search_roots
        for candidate in (
            root / "codigo" / "analisis_recurrencia.py",
            root / "Entrega_02" / "codigo" / "analisis_recurrencia.py",
        )
        if candidate.is_file()
    ),
    None,
)
if script_path is None:
    raise FileNotFoundError(f"No se encontró analisis_recurrencia.py desde {cwd}")

spec = importlib.util.spec_from_file_location("analisis_recurrencia", script_path)
if spec is None or spec.loader is None:
    raise ImportError(f"No se pudo cargar el módulo de análisis: {script_path}")
analysis = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = analysis
spec.loader.exec_module(analysis)
resultados = analysis.run_analysis()

print("Completez regional:")
print(resultados["completez"].query("Conjunto == 'Catálogo regional'").to_string(index=False))
print("\nRecurrencia regional:")
print(resultados["recurrencia"].query("Conjunto == 'Catálogo regional'").to_string(index=False))
print("\nAjustes Gutenberg-Richter:")
print(resultados["ajustes"].to_string(index=False))
print("\nComparación de ajustes crudos y corregidos:")
print(resultados["comparacion_ajustes"].to_string(index=False))

figuras_recurrencia = [
    analysis.DELIVERY_ROOT / "figuras" / f"curvas_recurrencia_sismofuentes_{escala}.png"
    for escala in ("local", "regional")
]
ajustes_regionales = resultados["ajustes"].query("Escala == 'Regional (260 km)' and Conjunto != 'Catálogo regional'")
if ajustes_regionales["a"].notna().sum() != 3 or not all(path.is_file() for path in figuras_recurrencia):
    raise RuntimeError("Completez y recurrencia no finalizaron con las tres sismofuentes regionales y las dos figuras consolidadas.")
print("\nCompletez y recurrencia local/regional finalizadas; se generaron las dos figuras consolidadas.")

## Curvas de atenuación exploratorias para Neiva

Esta etapa se ejecuta únicamente después de que la celda anterior termina la evaluación de completez por rangos, las tasas Gutenberg--Richter ajustadas y todas las gráficas de recurrencia local/regional. El módulo `analisis_atenuacion.py` aplica la relación de Zhao et al. (2006) a cada registro y calcula la distancia epicentral al Parque Santander. Las entradas que puede modificar están al comienzo del módulo en `CONFIG` y `MODEL_PARAMETERS`; también se exponen aquí antes de ejecutar el análisis.

**Suposiciones:** el punto de análisis usa las coordenadas de referencia definidas para el Parque Santander (2.9262767, -75.2892111); suelo medio se representa con la clase III y su término de sitio de Zhao; se usa PGA por defecto (la opción `Sa(0.5 s)` está disponible aparte); el campo `Z` se interpreta como profundidad focal en km, limitada a 0–125 km; las fuentes cortical, interplaca e intraplaca se asignan respectivamente a corteza, interfaz y slab. No se cuenta con mecanismos focales, por lo que `F_R=0`. Los residuales intra e interevento se fijan en cero, así que las predicciones son medianas condicionales, no percentiles ni acelerogramas. Zhao fue calibrada con sismos de Japón; los resultados para Neiva son exploratorios y no constituyen una GMPE local validada.

La distancia media por fuente y tramo se informa como resumen, pero la curva se calcula evento por evento con magnitud, distancia y profundidad de cada registro. Los percentiles de la tabla resumen describen dispersión entre eventos predichos; no representan la desviación estándar del modelo ni una tasa de amenaza.

In [ ]:
attenuation_script = next(
    (
        candidate
        for root in search_roots
        for candidate in (
            root / "codigo" / "analisis_atenuacion.py",
            root / "Entrega_02" / "codigo" / "analisis_atenuacion.py",
        )
        if candidate.is_file()
    ),
    None,
)
if attenuation_script is None:
    raise FileNotFoundError(f"No se encontró analisis_atenuacion.py desde {cwd}")

attenuation_spec = importlib.util.spec_from_file_location(
    "analisis_atenuacion", attenuation_script
)
if attenuation_spec is None or attenuation_spec.loader is None:
    raise ImportError(f"No se pudo cargar el módulo: {attenuation_script}")
attenuation = importlib.util.module_from_spec(attenuation_spec)
sys.modules[attenuation_spec.name] = attenuation
attenuation_spec.loader.exec_module(attenuation)

# Modifique estas entradas antes de ejecutar, si el escenario lo requiere.
configuracion_atenuacion = dict(attenuation.CONFIG)
configuracion_atenuacion["site_name"] = "Neiva - Parque Santander"
configuracion_atenuacion["site_latitude"] = 2.9262767
configuracion_atenuacion["site_longitude"] = -75.2892111
configuracion_atenuacion["site_class"] = "III"
configuracion_atenuacion["intensity_measure"] = "PGA"

resultados_atenuacion = attenuation.run_analysis(
    config=configuracion_atenuacion
)
print("Resumen por sismofuente y tramo de magnitud:")
print(resultados_atenuacion["resumen"].to_string(index=False))

## Archivos de salida

- `datos/datos estadistica/acumulado_anual_por_tramo_magnitud.csv`: N anual y N acumulado por año, intervalo, conjunto y escala.
- `datos/datos estadistica/completez_por_tramo_magnitud.csv`: años de inicio, exposición, pendientes y R² por tramo.
- `datos/datos estadistica/tabla_recurrencia_gr_por_tramo.csv`: conteos incrementales/acumulados, exposición, λ, log10(λ) y parámetros a, b, R².
- `datos/datos estadistica/ajuste_gutenberg_richter_sismofuentes.csv`: parámetros corregidos por completez, por escala y conjunto.
- `datos/datos estadistica/ajustes_crudos_y_corregidos_gutenberg_richter.csv`: comparación de parámetros a y b y R² de ambos ajustes.
- `datos/datos estadistica/recurrencia_sin_correccion_por_tramo.csv`: tasas anuales sin corregir, por tramo, escala y fuente.
- `datos/datos estadistica/conteos_incrementales_ventana_umbral.csv`: desglose de cada conteo acumulado por intervalo, en la ventana temporal del umbral.
- `datos/datos estadistica/predicciones_atenuacion_neiva_pga_por_evento.csv`: aceleración mediana condicional y distancia por evento.
- `datos/datos estadistica/resumen_atenuacion_neiva_pga_por_fuente_magnitud.csv`: resumen por sismofuente y tramo de magnitud.
- `datos/datos estadistica/parametros_atenuacion_neiva_pga.json`: parámetros y entradas utilizados para reproducir el escenario.
- `figuras/completez_acumulada_*.png`: acumulados anuales y regresiones segmentadas.
- `figuras/comparacion_recurrencia_cruda_completez_*.png`: comparación por sismofuente y escala, con ecuaciones y R².
- `figuras/curvas_recurrencia_sismofuentes_local.png` y `figuras/curvas_recurrencia_sismofuentes_regional.png`: curvas de tasas crudas/corregidas y ajustes para cada escala.
- `figuras/curvas_atenuacion_neiva_pga_fuentes_logx.png`: comparación consolidada de las tres sismofuentes en distancia logarítmica.
- Se conserva un gráfico de distancia lineal por sismofuente y se agrega, al lado, su versión con distancia en escala logarítmica.